In [19]:
import os
import numpy as np
import librosa
import tensorflow as tf
from sklearn.model_selection import train_test_split

# إعدادات الصوت والميزات
SR = 22050
FILE_DURATION = 30.0
SEGMENTS = 5
SEGMENT_DURATION = FILE_DURATION / SEGMENTS
IMG_SIZE = (224, 224)

def extract_segments(y_audio, sr):
    segment_len = int(SEGMENT_DURATION * sr)
    segments = []
    
    for i in range(SEGMENTS):
        start = i * segment_len
        end = start + segment_len
        if end > len(y_audio):
            break
        segments.append(y_audio[start:end])
    return segments

def process_and_split(data_path, genres, save_dir, feature_type='mfcc20', test_size=0.2):
    X, y = [], []
    
    # Determine number of MFCCs if using MFCC
    N_MFCC = 20
    if feature_type == 'mfcc13':
        N_MFCC = 13
    elif feature_type == 'mfcc20':
        N_MFCC = 20
    elif feature_type == 'mfcc40':
        N_MFCC = 40
    
    for idx, genre in enumerate(genres):
        print(f"Processing Genre: {genre}")
        genre_path = os.path.join(data_path, genre)
        for filename in os.listdir(genre_path):
            if filename.lower().endswith(".wav"):
                file_path = os.path.join(genre_path, filename)
                y_audio, _ = librosa.load(file_path, sr=SR, duration=FILE_DURATION)
                
                segments = extract_segments(y_audio, SR)
                
                for seg in segments:
                    if feature_type.startswith('mfcc'):
                        feature = librosa.feature.mfcc(y=seg, sr=SR, n_mfcc=N_MFCC)
                    elif feature_type == 'chroma':
                        feature = librosa.feature.chroma_stft(y=seg, sr=SR)
                    elif feature_type == 'mel_spectrogram':
                        mel = librosa.feature.melspectrogram(y=seg, sr=SR)
                        feature = librosa.power_to_db(mel)
                    else:
                        raise ValueError("Unknown feature type")
                    
                    feature = tf.image.resize(feature[..., np.newaxis], IMG_SIZE).numpy()
                    feature = np.repeat(feature, 3, axis=-1)

                    X.append(feature)
                    y.append(idx)

    X = np.array(X)
    y = np.array(y)

    print(f"\nTotal Samples After Segmentation: {len(X)}")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42, stratify=y
    )

    # Create save directory
    train_dir = os.path.join(save_dir, "train", feature_type)
    test_dir = os.path.join(save_dir, "test", feature_type)
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)

    np.save(os.path.join(train_dir, f"{feature_type}_X_train.npy"), X_train)
    np.save(os.path.join(train_dir, f"{feature_type}_y_train.npy"), y_train)
    np.save(os.path.join(test_dir, f"{feature_type}_X_test.npy"), X_test)
    np.save(os.path.join(test_dir, f"{feature_type}_y_test.npy"), y_test)

    print(f"\n=== Final Data Information for Feature: {feature_type} ===")
    print("Training Data Shape:", X_train.shape)
    print("Training Labels Shape:", y_train.shape)
    print("Testing Data Shape:", X_test.shape)
    print("Testing Labels Shape:", y_test.shape)

# ==========================
# Main execution
# ==========================
if __name__ == "__main__":
    # Ask user to choose feature
    features = ['mfcc13', 'mfcc20', 'mfcc40', 'chroma', 'mel_spectrogram']
    print("Available Features:")
    for i, feat in enumerate(features, 1):
        print(f"  {i}. {feat}")
    
    choice = input("Select feature by number (1-5): ").strip()
    try:
        feature_type = features[int(choice)-1]
    except:
        feature_type = 'mfcc20'  # default
    
    data_path = "/root/YMED/"
    genres = ['adeni', 'hadrami', 'lahji', 'sanaai', 'tihami']
    save_dir = "/root/autodl-tmp/split_data/"

    process_and_split(data_path, genres, save_dir, feature_type=feature_type, test_size=0.2)


Available Features:
  1. mfcc13
  2. mfcc20
  3. mfcc40
  4. chroma
  5. mel_spectrogram


Select feature by number (1-5):  1


Processing Genre: adeni


2026-05-02 17:25:44.766635: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-02 17:25:46.350885: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1532] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 10418 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1080 Ti, pci bus id: 0000:09:00.0, compute capability: 6.1


Processing Genre: hadrami
Processing Genre: lahji
Processing Genre: sanaai
Processing Genre: tihami

Total Samples After Segmentation: 5653

=== Final Data Information for Feature: mfcc13 ===
Training Data Shape: (4522, 224, 224, 3)
Training Labels Shape: (4522,)
Testing Data Shape: (1131, 224, 224, 3)
Testing Labels Shape: (1131,)


In [1]:
import os
import csv
import numpy as np
import librosa
import tensorflow as tf
from sklearn.model_selection import train_test_split

# ==========================
# Audio and feature settings
# ==========================
SR = 22050
FILE_DURATION = 30.0
SEGMENTS = 5
SEGMENT_DURATION = FILE_DURATION / SEGMENTS
IMG_SIZE = (224, 224)
RANDOM_STATE = 42


def load_audio_30s(file_path, sr=SR, duration=FILE_DURATION):
    """
    Load audio, resample it, and force it to exactly 30 seconds.
    If the file is shorter than 30 seconds, it is zero-padded.
    If it is longer, it is truncated.
    """
    target_len = int(sr * duration)
    y_audio, _ = librosa.load(file_path, sr=sr, mono=True)

    if len(y_audio) < target_len:
        y_audio = np.pad(y_audio, (0, target_len - len(y_audio)), mode="constant")
    else:
        y_audio = y_audio[:target_len]

    return y_audio


def extract_segments(y_audio, sr=SR):
    """
    Split one 30-second audio signal into five non-overlapping 6-second segments.
    """
    segment_len = int(SEGMENT_DURATION * sr)
    segments = []

    for i in range(SEGMENTS):
        start = i * segment_len
        end = start + segment_len
        segment = y_audio[start:end]

        if len(segment) == segment_len:
            segments.append(segment)

    return segments


def apply_audio_augmentation(seg, sr=SR):
    """
    Audio-level augmentation.
    These augmentations must be applied only to training data.
    """
    augmented_segments = []

    # Original segment
    augmented_segments.append(("original", seg))

    # Time shifting
    shift = int(0.1 * len(seg))
    shifted = np.roll(seg, shift)
    augmented_segments.append(("time_shift", shifted))

    # Pitch shifting
    try:
        pitched = librosa.effects.pitch_shift(y=seg, sr=sr, n_steps=2)
        augmented_segments.append(("pitch_shift", pitched))
    except Exception:
        pass

    # Noise injection
    noise = 0.005 * np.random.randn(len(seg))
    noisy = seg + noise
    augmented_segments.append(("noise", noisy))

    return augmented_segments


def spec_augment(feature, max_freq_mask=12, max_time_mask=20):
    """
    Simple SpecAugment applied on the 2D feature map.
    This should be applied only to training features.
    """
    feature = feature.copy()

    num_freq, num_time = feature.shape

    # Frequency masking
    f = np.random.randint(0, min(max_freq_mask, num_freq) + 1)
    if f > 0:
        f0 = np.random.randint(0, max(1, num_freq - f))
        feature[f0:f0 + f, :] = 0

    # Time masking
    t = np.random.randint(0, min(max_time_mask, num_time) + 1)
    if t > 0:
        t0 = np.random.randint(0, max(1, num_time - t))
        feature[:, t0:t0 + t] = 0

    return feature


def extract_feature(seg, feature_type="mfcc20", sr=SR):
    """
    Extract one feature representation from one 6-second segment.
    """
    if feature_type == "mfcc13":
        feature = librosa.feature.mfcc(y=seg, sr=sr, n_mfcc=13)

    elif feature_type == "mfcc20":
        feature = librosa.feature.mfcc(y=seg, sr=sr, n_mfcc=20)

    elif feature_type == "mfcc40":
        feature = librosa.feature.mfcc(y=seg, sr=sr, n_mfcc=40)

    elif feature_type == "chroma":
        feature = librosa.feature.chroma_stft(y=seg, sr=sr)

    elif feature_type == "mel_spectrogram":
        mel = librosa.feature.melspectrogram(y=seg, sr=sr, n_mels=128)
        feature = librosa.power_to_db(mel, ref=np.max)

    elif feature_type == "filterbank":
        mel = librosa.feature.melspectrogram(y=seg, sr=sr, n_mels=40)
        feature = librosa.power_to_db(mel, ref=np.max)

    else:
        raise ValueError(f"Unknown feature type: {feature_type}")

    return feature.astype(np.float32)


def prepare_feature_image(feature):
    """
    Resize feature map to 224x224 and convert it to 3 channels.
    """
    feature = tf.image.resize(feature[..., np.newaxis], IMG_SIZE).numpy()
    feature = np.repeat(feature, 3, axis=-1)
    return feature.astype(np.float32)


def collect_original_files(data_path, genres):
    """
    Collect original 30-second files before segmentation.
    This is essential to avoid data leakage.
    """
    file_paths = []
    labels = []
    genre_names = []

    for idx, genre in enumerate(genres):
        genre_path = os.path.join(data_path, genre)

        if not os.path.isdir(genre_path):
            raise FileNotFoundError(f"Genre folder not found: {genre_path}")

        wav_files = sorted([
            f for f in os.listdir(genre_path)
            if f.lower().endswith(".wav")
        ])

        print(f"{genre}: {len(wav_files)} original files")

        for filename in wav_files:
            file_paths.append(os.path.join(genre_path, filename))
            labels.append(idx)
            genre_names.append(genre)

    return np.array(file_paths), np.array(labels), np.array(genre_names)


def process_split(file_paths, labels, split_name, feature_type, augment_train=False):
    """
    Process one split only.
    Segmentation happens after the original files have already been assigned
    to train or test.
    """
    X, y = [], []
    metadata = []

    for file_path, label in zip(file_paths, labels):
        filename = os.path.basename(file_path)

        try:
            y_audio = load_audio_30s(file_path)
            segments = extract_segments(y_audio)

            for seg_id, seg in enumerate(segments):
                if split_name == "train" and augment_train:
                    augmented_segments = apply_audio_augmentation(seg)
                else:
                    augmented_segments = [("original", seg)]

                for aug_name, aug_seg in augmented_segments:
                    feature = extract_feature(
                        aug_seg,
                        feature_type=feature_type,
                        sr=SR
                    )

                    if split_name == "train" and augment_train and aug_name == "original":
                        # Optional SpecAugment only on original training feature
                        feature = spec_augment(feature)

                    feature_img = prepare_feature_image(feature)

                    X.append(feature_img)
                    y.append(label)

                    metadata.append({
                        "split": split_name,
                        "original_file": filename,
                        "segment_id": seg_id,
                        "augmentation": aug_name,
                        "label": int(label)
                    })

        except Exception as e:
            print(f"Skipping file due to error: {file_path}")
            print(f"Error: {e}")

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64), metadata


def save_metadata(metadata, save_path):
    """
    Save metadata to verify that no original file appears in both train and test.
    """
    if len(metadata) == 0:
        return

    keys = metadata[0].keys()

    with open(save_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(metadata)


def check_no_data_leakage(train_files, test_files):
    """
    Confirm that no original 30-second file exists in both train and test.
    """
    train_set = set([os.path.basename(f) for f in train_files])
    test_set = set([os.path.basename(f) for f in test_files])

    overlap = train_set.intersection(test_set)

    if len(overlap) > 0:
        raise RuntimeError(
            f"Data leakage detected! Overlapping files: {list(overlap)[:10]}"
        )

    print("\nData leakage check: PASSED")
    print("No original 30-second audio file appears in both train and test.")


def process_and_split(
    data_path,
    genres,
    save_dir,
    feature_type="mel_spectrogram",
    test_size=0.2,
    augment_train=False
):
    """
    Correct pipeline:
    1. Collect original 30-second files.
    2. Split original files into train/test.
    3. Segment files inside each split only.
    4. Extract features.
    5. Save train/test arrays.
    """

    print("\nCollecting original audio files...")
    file_paths, labels, genre_names = collect_original_files(data_path, genres)

    print("\nTotal original files:", len(file_paths))

    # Split first at the original-file level
    train_files, test_files, train_labels, test_labels = train_test_split(
        file_paths,
        labels,
        test_size=test_size,
        random_state=RANDOM_STATE,
        stratify=labels
    )

    check_no_data_leakage(train_files, test_files)

    print("\nOriginal-file split:")
    print("Train original files:", len(train_files))
    print("Test original files:", len(test_files))

    print(f"\nProcessing TRAIN split for feature: {feature_type}")
    X_train, y_train, train_metadata = process_split(
        train_files,
        train_labels,
        split_name="train",
        feature_type=feature_type,
        augment_train=augment_train
    )

    print(f"\nProcessing TEST split for feature: {feature_type}")
    X_test, y_test, test_metadata = process_split(
        test_files,
        test_labels,
        split_name="test",
        feature_type=feature_type,
        augment_train=False
    )

    # Save directories
    train_dir = os.path.join(save_dir, "train", feature_type)
    test_dir = os.path.join(save_dir, "test", feature_type)
    meta_dir = os.path.join(save_dir, "metadata", feature_type)

    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)
    os.makedirs(meta_dir, exist_ok=True)

    # Save arrays
    np.save(os.path.join(train_dir, f"{feature_type}_X_train.npy"), X_train)
    np.save(os.path.join(train_dir, f"{feature_type}_y_train.npy"), y_train)

    np.save(os.path.join(test_dir, f"{feature_type}_X_test.npy"), X_test)
    np.save(os.path.join(test_dir, f"{feature_type}_y_test.npy"), y_test)

    # Save metadata
    save_metadata(
        train_metadata,
        os.path.join(meta_dir, f"{feature_type}_train_metadata.csv")
    )
    save_metadata(
        test_metadata,
        os.path.join(meta_dir, f"{feature_type}_test_metadata.csv")
    )

    # Save original split file list
    with open(os.path.join(meta_dir, f"{feature_type}_original_split.csv"), "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["split", "file_path", "label"])

        for fp, lb in zip(train_files, train_labels):
            writer.writerow(["train", fp, int(lb)])

        for fp, lb in zip(test_files, test_labels):
            writer.writerow(["test", fp, int(lb)])

    print(f"\n=== Final Data Information for Feature: {feature_type} ===")
    print("Training Data Shape:", X_train.shape)
    print("Training Labels Shape:", y_train.shape)
    print("Testing Data Shape:", X_test.shape)
    print("Testing Labels Shape:", y_test.shape)

    print("\nExpected sample count without augmentation:")
    print(f"Train expected approximately: {len(train_files)} × {SEGMENTS} = {len(train_files) * SEGMENTS}")
    print(f"Test expected approximately: {len(test_files)} × {SEGMENTS} = {len(test_files) * SEGMENTS}")

    if augment_train:
        print("\nNote: Training sample count is larger because augmentation was applied only to training data.")

    print("\nSaved successfully.")


# ==========================
# Main execution
# ==========================
if __name__ == "__main__":

    features = [
        "mfcc13",
        "mfcc20",
        "mfcc40",
        "chroma",
        "mel_spectrogram",
        "filterbank"
    ]

    print("Available Features:")
    for i, feat in enumerate(features, 1):
        print(f"  {i}. {feat}")

    choice = input("Select feature by number (1-6): ").strip()

    try:
        feature_type = features[int(choice) - 1]
    except Exception:
        print("Invalid choice. Using default feature: mfcc20")
        feature_type = "mfcc20"

    data_path = "/root/YMED/"
    genres = ["adeni", "hadrami", "lahji", "sanaai", "tihami"]
    save_dir = "/root/autodl-tmp/split_data/"

    # Set True if you want augmentation for reviewer response.
    # Augmentation is applied only to training data.
    augment_train = False

    process_and_split(
        data_path=data_path,
        genres=genres,
        save_dir=save_dir,
        feature_type=feature_type,
        test_size=0.2,
        augment_train=augment_train
    )

Available Features:
  1. mfcc13
  2. mfcc20
  3. mfcc40
  4. chroma
  5. mel_spectrogram
  6. filterbank


Select feature by number (1-6):  1



adeni: 230 original files
hadrami: 230 original files
lahji: 230 original files
sanaai: 230 original files
tihami: 230 original files

Total original files: 1150

Data leakage check: PASSED
No original 30-second audio file appears in both train and test.

Original-file split:
Train original files: 920
Test original files: 230

Processing TRAIN split for feature: mfcc13


2026-05-03 15:32:50.919897: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-03 15:32:51.561102: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1532] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 30416 MB memory:  -> device: 0, name: NVIDIA vGPU-32GB, pci bus id: 0000:82:00.0, compute capability: 8.9



Processing TEST split for feature: mfcc13

=== Final Data Information for Feature: mfcc13 ===
Training Data Shape: (4600, 224, 224, 3)
Training Labels Shape: (4600,)
Testing Data Shape: (1150, 224, 224, 3)
Testing Labels Shape: (1150,)

Expected sample count without augmentation:
Train expected approximately: 920 × 5 = 4600
Test expected approximately: 230 × 5 = 1150

Saved successfully.


In [1]:
import os
import csv
import numpy as np
import librosa
import tensorflow as tf
from sklearn.model_selection import train_test_split


# ============================================================
# Audio and feature settings
# ============================================================

SR = 22050
FILE_DURATION = 30.0
SEGMENTS = 5
SEGMENT_DURATION = FILE_DURATION / SEGMENTS
IMG_SIZE = (224, 224)
RANDOM_STATE = 42


# ============================================================
# Basic audio loading and segmentation
# ============================================================

def load_audio_30s(file_path, sr=SR, duration=FILE_DURATION):
    """
    Load audio, resample it, convert it to mono, and force it to exactly 30 seconds.
    If the file is shorter than 30 seconds, it is zero-padded.
    If the file is longer than 30 seconds, it is truncated.
    """
    target_len = int(sr * duration)

    y_audio, _ = librosa.load(file_path, sr=sr, mono=True)

    if len(y_audio) < target_len:
        y_audio = np.pad(y_audio, (0, target_len - len(y_audio)), mode="constant")
    else:
        y_audio = y_audio[:target_len]

    return y_audio.astype(np.float32)


def fix_length_audio(y, target_len):
    """
    Crop or pad audio to a fixed target length.
    This is needed after time stretching.
    """
    if len(y) > target_len:
        y = y[:target_len]
    elif len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)), mode="constant")

    return y.astype(np.float32)


def extract_segments(y_audio, sr=SR):
    """
    Split one 30-second audio signal into five non-overlapping 6-second segments.
    """
    segment_len = int(SEGMENT_DURATION * sr)
    segments = []

    for i in range(SEGMENTS):
        start = i * segment_len
        end = start + segment_len
        segment = y_audio[start:end]

        if len(segment) == segment_len:
            segments.append(segment.astype(np.float32))

    return segments


# ============================================================
# Waveform-level augmentation BEFORE feature extraction
# ============================================================

def augment_waveform_segment(seg, sr=SR, seed=None):
    """
    Apply audio-level augmentation BEFORE feature extraction.

    This function returns a list of augmented waveform segments:
    - original
    - time_shift
    - pitch_shift
    - time_stretch
    - noise_injection

    Important:
    Use this only for training data.
    Do not use it for test data.
    """
    rng = np.random.default_rng(seed)
    target_len = len(seg)

    augmented_segments = []

    # 1. Original segment
    augmented_segments.append(("original", seg.astype(np.float32)))

    # 2. Time shifting
    try:
        max_shift = int(0.10 * target_len)
        shift = int(rng.integers(-max_shift, max_shift + 1))
        shifted = np.roll(seg, shift)
        augmented_segments.append(("time_shift", shifted.astype(np.float32)))
    except Exception as e:
        print(f"Time shift failed: {e}")

    # 3. Pitch shifting
    try:
        n_steps = float(rng.choice([-2, -1, 1, 2]))
        pitched = librosa.effects.pitch_shift(y=seg, sr=sr, n_steps=n_steps)
        pitched = fix_length_audio(pitched, target_len)
        augmented_segments.append(("pitch_shift", pitched.astype(np.float32)))
    except Exception as e:
        print(f"Pitch shift failed: {e}")

    # 4. Time stretching
    try:
        rate = float(rng.uniform(0.90, 1.10))
        stretched = librosa.effects.time_stretch(y=seg, rate=rate)
        stretched = fix_length_audio(stretched, target_len)
        augmented_segments.append(("time_stretch", stretched.astype(np.float32)))
    except Exception as e:
        print(f"Time stretch failed: {e}")

    # 5. Noise injection
    try:
        noise_level = float(rng.uniform(0.003, 0.008))
        noise = noise_level * rng.standard_normal(target_len)
        noisy = seg + noise
        noisy = fix_length_audio(noisy, target_len)
        augmented_segments.append(("noise_injection", noisy.astype(np.float32)))
    except Exception as e:
        print(f"Noise injection failed: {e}")

    return augmented_segments


# ============================================================
# Feature extraction
# ============================================================

def extract_feature(seg, feature_type="mfcc20", sr=SR):
    """
    Extract one feature representation from one 6-second audio segment.
    """
    if feature_type == "mfcc13":
        feature = librosa.feature.mfcc(y=seg, sr=sr, n_mfcc=13)

    elif feature_type == "mfcc20":
        feature = librosa.feature.mfcc(y=seg, sr=sr, n_mfcc=20)

    elif feature_type == "mfcc40":
        feature = librosa.feature.mfcc(y=seg, sr=sr, n_mfcc=40)

    elif feature_type == "chroma":
        feature = librosa.feature.chroma_stft(y=seg, sr=sr)

    elif feature_type == "mel_spectrogram":
        mel = librosa.feature.melspectrogram(y=seg, sr=sr, n_mels=128)
        feature = librosa.power_to_db(mel, ref=np.max)

    elif feature_type == "filterbank":
        mel = librosa.feature.melspectrogram(y=seg, sr=sr, n_mels=40)
        feature = librosa.power_to_db(mel, ref=np.max)

    else:
        raise ValueError(f"Unknown feature type: {feature_type}")

    return feature.astype(np.float32)


# ============================================================
# SpecAugment AFTER feature extraction
# ============================================================

def spec_augment(feature, max_freq_mask=12, max_time_mask=20, seed=None):
    """
    Apply SpecAugment to a 2D feature map after feature extraction.

    This should be applied only to training features.
    It should not be applied to test features.
    """
    rng = np.random.default_rng(seed)
    feature_aug = feature.copy()

    if feature_aug.ndim != 2:
        raise ValueError("SpecAugment expects a 2D feature map.")

    num_freq, num_time = feature_aug.shape

    # Frequency masking
    max_f = min(max_freq_mask, num_freq)
    if max_f > 0:
        f = int(rng.integers(0, max_f + 1))
        if f > 0 and num_freq - f > 0:
            f0 = int(rng.integers(0, num_freq - f + 1))
            feature_aug[f0:f0 + f, :] = 0.0

    # Time masking
    max_t = min(max_time_mask, num_time)
    if max_t > 0:
        t = int(rng.integers(0, max_t + 1))
        if t > 0 and num_time - t > 0:
            t0 = int(rng.integers(0, num_time - t + 1))
            feature_aug[:, t0:t0 + t] = 0.0

    return feature_aug.astype(np.float32)


def maybe_apply_specaugment(feature, apply_specaug=False, seed=None):
    """
    Apply SpecAugment only if requested.
    """
    if apply_specaug:
        return spec_augment(feature, seed=seed)
    return feature


# ============================================================
# Prepare feature as image input for CNN models
# ============================================================

def prepare_feature_image(feature):
    """
    Resize feature map to 224x224 and convert it to 3 channels.
    Output shape: (224, 224, 3)
    """
    feature = feature.astype(np.float32)

    feature_img = tf.image.resize(
        feature[..., np.newaxis],
        IMG_SIZE
    ).numpy()

    feature_img = np.repeat(feature_img, 3, axis=-1)

    return feature_img.astype(np.float32)


# ============================================================
# Dataset collection and leakage check
# ============================================================

def collect_original_files(data_path, genres):
    """
    Collect original 30-second WAV files before segmentation.

    This is essential to avoid data leakage:
    split must happen at the original-file level.
    """
    file_paths = []
    labels = []
    genre_names = []

    for idx, genre in enumerate(genres):
        genre_path = os.path.join(data_path, genre)

        if not os.path.isdir(genre_path):
            raise FileNotFoundError(f"Genre folder not found: {genre_path}")

        wav_files = sorted([
            f for f in os.listdir(genre_path)
            if f.lower().endswith(".wav")
        ])

        print(f"{genre}: {len(wav_files)} original files")

        for filename in wav_files:
            full_path = os.path.join(genre_path, filename)
            file_paths.append(full_path)
            labels.append(idx)
            genre_names.append(genre)

    return np.array(file_paths), np.array(labels), np.array(genre_names)


def check_no_data_leakage(train_files, test_files):
    """
    Confirm that no original 30-second file exists in both train and test.

    Uses normalized full paths to avoid false overlap when two genres contain
    files with similar names.
    """
    train_set = set([os.path.abspath(f) for f in train_files])
    test_set = set([os.path.abspath(f) for f in test_files])

    overlap = train_set.intersection(test_set)

    if len(overlap) > 0:
        raise RuntimeError(
            f"Data leakage detected! Overlapping files: {list(overlap)[:10]}"
        )

    print("\nData leakage check: PASSED")
    print("No original 30-second audio file appears in both train and test.")


# ============================================================
# Metadata saving
# ============================================================

def save_metadata(metadata, save_path):
    """
    Save metadata to CSV.
    This helps verify train/test separation and augmentation details.
    """
    if len(metadata) == 0:
        print(f"No metadata to save for: {save_path}")
        return

    keys = metadata[0].keys()

    with open(save_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(metadata)


# ============================================================
# Process one split
# ============================================================

def process_split(
    file_paths,
    labels,
    split_name,
    feature_type,
    augment_train=False,
    apply_specaug=True,
    random_state=RANDOM_STATE
):
    """
    Process one split only.

    Correct reviewer-safe pipeline:
    1. Files are already assigned to train/test.
    2. Each file is loaded and segmented.
    3. Augmentation is applied only if split is train.
    4. Features are extracted.
    5. SpecAugment is applied only to training features.
    6. Feature maps are resized for CNN input.
    """
    X = []
    y = []
    metadata = []

    rng = np.random.default_rng(random_state)

    for file_index, (file_path, label) in enumerate(zip(file_paths, labels)):
        filename = os.path.basename(file_path)

        try:
            y_audio = load_audio_30s(file_path)
            segments = extract_segments(y_audio)

            if len(segments) != SEGMENTS:
                print(f"Warning: {filename} produced {len(segments)} segments instead of {SEGMENTS}")

            for seg_id, seg in enumerate(segments):

                # Training: use original + augmented waveforms
                if split_name == "train" and augment_train:
                    seg_seed = int(rng.integers(0, 1_000_000_000))
                    waveform_versions = augment_waveform_segment(
                        seg,
                        sr=SR,
                        seed=seg_seed
                    )
                else:
                    waveform_versions = [("original", seg)]

                for aug_name, aug_seg in waveform_versions:

                    feature = extract_feature(
                        aug_seg,
                        feature_type=feature_type,
                        sr=SR
                    )

                    # SpecAugment after feature extraction, train only
                    used_specaug = False
                    if split_name == "train" and augment_train and apply_specaug:
                        spec_seed = int(rng.integers(0, 1_000_000_000))
                        feature = maybe_apply_specaugment(
                            feature,
                            apply_specaug=True,
                            seed=spec_seed
                        )
                        used_specaug = True

                    feature_img = prepare_feature_image(feature)

                    X.append(feature_img)
                    y.append(label)

                    metadata.append({
                        "split": split_name,
                        "original_file": filename,
                        "full_path": os.path.abspath(file_path),
                        "segment_id": int(seg_id),
                        "augmentation": aug_name,
                        "specaugment": int(used_specaug),
                        "label": int(label)
                    })

        except Exception as e:
            print(f"Skipping file due to error: {file_path}")
            print(f"Error: {e}")

    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.int64)

    return X, y, metadata


# ============================================================
# Main processing function
# ============================================================

def process_and_split(
    data_path,
    genres,
    save_dir,
    feature_type="mfcc13",
    test_size=0.2,
    augment_train=True,
    apply_specaug=True,
    random_state=RANDOM_STATE,
    force_recreate=True
):
    """
    Full corrected pipeline for reviewer response.

    Steps:
    1. Collect original 30-second files.
    2. Split original files into train/test.
    3. Check no data leakage.
    4. Segment files inside each split only.
    5. Apply waveform augmentation only to train.
    6. Extract features.
    7. Apply SpecAugment only to train features.
    8. Save arrays and metadata.
    """

    print("\n" + "=" * 80)
    print(f"Starting processing for feature: {feature_type}")
    print("=" * 80)

    train_dir = os.path.join(save_dir, "train", feature_type)
    test_dir = os.path.join(save_dir, "test", feature_type)
    meta_dir = os.path.join(save_dir, "metadata", feature_type)

    x_train_path = os.path.join(train_dir, f"{feature_type}_X_train.npy")
    y_train_path = os.path.join(train_dir, f"{feature_type}_y_train.npy")
    x_test_path = os.path.join(test_dir, f"{feature_type}_X_test.npy")
    y_test_path = os.path.join(test_dir, f"{feature_type}_y_test.npy")

    if (
        os.path.exists(x_train_path)
        and os.path.exists(y_train_path)
        and os.path.exists(x_test_path)
        and os.path.exists(y_test_path)
        and not force_recreate
    ):
        print("Existing processed files found. Loading saved arrays...")
        X_train = np.load(x_train_path)
        y_train = np.load(y_train_path)
        X_test = np.load(x_test_path)
        y_test = np.load(y_test_path)

        print("Loaded successfully:")
        print("X_train:", X_train.shape)
        print("y_train:", y_train.shape)
        print("X_test:", X_test.shape)
        print("y_test:", y_test.shape)

        return X_train, X_test, y_train, y_test

    print("\nCollecting original audio files...")
    file_paths, labels, genre_names = collect_original_files(data_path, genres)

    print("\nTotal original files:", len(file_paths))
    print("Total labels:", len(labels))

    # Split first at the original-file level
    train_files, test_files, train_labels, test_labels = train_test_split(
        file_paths,
        labels,
        test_size=test_size,
        random_state=random_state,
        stratify=labels
    )

    check_no_data_leakage(train_files, test_files)

    print("\nOriginal-file split:")
    print("Train original files:", len(train_files))
    print("Test original files:", len(test_files))

    print("\nExpected sample count without augmentation:")
    print(f"Train: {len(train_files)} × {SEGMENTS} = {len(train_files) * SEGMENTS}")
    print(f"Test : {len(test_files)} × {SEGMENTS} = {len(test_files) * SEGMENTS}")

    print(f"\nProcessing TRAIN split for feature: {feature_type}")
    print(f"Waveform augmentation enabled: {augment_train}")
    print(f"SpecAugment enabled: {apply_specaug}")

    X_train, y_train, train_metadata = process_split(
        train_files,
        train_labels,
        split_name="train",
        feature_type=feature_type,
        augment_train=augment_train,
        apply_specaug=apply_specaug,
        random_state=random_state
    )

    print(f"\nProcessing TEST split for feature: {feature_type}")
    print("No augmentation is applied to the test set.")

    X_test, y_test, test_metadata = process_split(
        test_files,
        test_labels,
        split_name="test",
        feature_type=feature_type,
        augment_train=False,
        apply_specaug=False,
        random_state=random_state
    )

    # Create directories
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)
    os.makedirs(meta_dir, exist_ok=True)

    # Save arrays
    np.save(x_train_path, X_train)
    np.save(y_train_path, y_train)
    np.save(x_test_path, X_test)
    np.save(y_test_path, y_test)

    # Save metadata
    save_metadata(
        train_metadata,
        os.path.join(meta_dir, f"{feature_type}_train_metadata.csv")
    )

    save_metadata(
        test_metadata,
        os.path.join(meta_dir, f"{feature_type}_test_metadata.csv")
    )

    # Save original split list
    split_csv_path = os.path.join(meta_dir, f"{feature_type}_original_split.csv")
    with open(split_csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["split", "file_path", "label"])

        for fp, lb in zip(train_files, train_labels):
            writer.writerow(["train", os.path.abspath(fp), int(lb)])

        for fp, lb in zip(test_files, test_labels):
            writer.writerow(["test", os.path.abspath(fp), int(lb)])

    print("\n" + "=" * 80)
    print(f"Final Data Information for Feature: {feature_type}")
    print("=" * 80)
    print("Training Data Shape:", X_train.shape)
    print("Training Labels Shape:", y_train.shape)
    print("Testing Data Shape:", X_test.shape)
    print("Testing Labels Shape:", y_test.shape)

    print("\nSaved files:")
    print(x_train_path)
    print(y_train_path)
    print(x_test_path)
    print(y_test_path)
    print(split_csv_path)

    if augment_train:
        print("\nNote:")
        print("Training sample count is larger because waveform-level augmentation")
        print("was applied before feature extraction only to the training set.")

    print("\nProcessing completed successfully.")

    return X_train, X_test, y_train, y_test


# ============================================================
# Run for one feature or all features
# ============================================================

def run_feature_extraction_pipeline(
    data_path,
    genres,
    save_dir,
    features=None,
    process_all=True,
    selected_feature="mfcc13",
    augment_train=True,
    apply_specaug=True,
    test_size=0.2,
    random_state=RANDOM_STATE,
    force_recreate=True
):
    """
    Run the full pipeline for either one feature or all features.
    """

    if features is None:
        features = [
            "mfcc13",
            "mfcc20",
            "mfcc40",
            "chroma",
            "mel_spectrogram",
            "filterbank"
        ]

    if process_all:
        selected_features = features
    else:
        selected_features = [selected_feature]

    outputs = {}

    for feature_type in selected_features:
        X_train, X_test, y_train, y_test = process_and_split(
            data_path=data_path,
            genres=genres,
            save_dir=save_dir,
            feature_type=feature_type,
            test_size=test_size,
            augment_train=augment_train,
            apply_specaug=apply_specaug,
            random_state=random_state,
            force_recreate=force_recreate
        )

        outputs[feature_type] = {
            "X_train": X_train,
            "X_test": X_test,
            "y_train": y_train,
            "y_test": y_test
        }

    return outputs


# ============================================================
# Configuration: edit these paths only
# ============================================================

data_path = "/root/YMED/"
genres = ["adeni", "hadrami", "lahji", "sanaai", "tihami"]
save_dir = "/root/autodl-tmp/split_data/"

features = [
    "mfcc13"
    #"mfcc20",
   # "mfcc40",
    #"chrom",
    #"mel_spectrogram"
    #"filterbank"
]


# ============================================================
# Option 1: Process all features
# ============================================================

outputs = run_feature_extraction_pipeline(
    data_path=data_path,
    genres=genres,
    save_dir=save_dir,
    features=features,
    process_all=True,
    selected_feature="filterbank",
    augment_train=True,
    apply_specaug=True,
    test_size=0.2,
    random_state=42,
    force_recreate=True
)


# ============================================================
# Option 2: If you want only one feature, comment Option 1 above
# and uncomment this block.
# ============================================================

# outputs = run_feature_extraction_pipeline(
#     data_path=data_path,
#     genres=genres,
#     save_dir=save_dir,
#     features=features,
#     process_all=False,
#     selected_feature="mel_spectrogram",
#     augment_train=True,
#     apply_specaug=True,
#     test_size=0.2,
#     random_state=42,
#     force_recreate=True
# )


Starting processing for feature: mfcc13

adeni: 230 original files
hadrami: 230 original files
lahji: 230 original files
sanaai: 230 original files
tihami: 230 original files

Total original files: 1150
Total labels: 1150

Data leakage check: PASSED
No original 30-second audio file appears in both train and test.

Original-file split:
Train original files: 920
Test original files: 230

Expected sample count without augmentation:
Train: 920 × 5 = 4600
Test : 230 × 5 = 1150

Processing TRAIN split for feature: mfcc13
Waveform augmentation enabled: True
SpecAugment enabled: True


2026-05-10 13:30:29.796236: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-10 13:30:30.292148: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1532] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 30416 MB memory:  -> device: 0, name: NVIDIA vGPU-32GB, pci bus id: 0000:81:00.0, compute capability: 8.9



Processing TEST split for feature: mfcc13
No augmentation is applied to the test set.

Final Data Information for Feature: mfcc13
Training Data Shape: (23000, 224, 224, 3)
Training Labels Shape: (23000,)
Testing Data Shape: (1150, 224, 224, 3)
Testing Labels Shape: (1150,)

Saved files:
/root/autodl-tmp/split_data/train/mfcc13/mfcc13_X_train.npy
/root/autodl-tmp/split_data/train/mfcc13/mfcc13_y_train.npy
/root/autodl-tmp/split_data/test/mfcc13/mfcc13_X_test.npy
/root/autodl-tmp/split_data/test/mfcc13/mfcc13_y_test.npy
/root/autodl-tmp/split_data/metadata/mfcc13/mfcc13_original_split.csv

Note:
Training sample count is larger because waveform-level augmentation
was applied before feature extraction only to the training set.

Processing completed successfully.
